In [ ]:
import models.utils as utils 
import models.agents.uni_random as uni_random
import models.agents.heuristics as heuristics
import models.agents.mcts as mcts
import analysis.analysis_utils as analysis_utils
import matplotlib.pylab as plt 
import seaborn as sns 
import itertools 
import pandas as pd
from tqdm import tqdm
from tqdm import trange
import random
import multiprocessing
import models.just_think.run_models 

import datetime
import importlib 
from itertools import chain
import constants
import numpy as np
import json

%matplotlib inline


save_dir = "play_figs/"
main_final_save_dir = "final_processed_res/play/"

In [ ]:
""" 
Init constants
"""
importlib.reload(constants)
model2name = constants.MODEL2NAME
model2pth = constants.MODEL2PTH["think"]

ordered_models = ["ours", "random",]

"""
Load in games, and filter to include just the games from the live gameplay study
"""
games, idx2game, game2idx, game_stimuli = analysis_utils.process_game_stimuli(constants.THINK_STIMULI_PTH)
livegameplay_games = pd.read_csv(constants.PLAY_STIMULI_PTH)
livegameplay_games["stimuli_id"] = [analysis_utils.get_stimuli_id(row) for idx, row in livegameplay_games.iterrows()]
play_game_objs = []
played_game_ids = set()
for game_id in livegameplay_games["stimuli_id"]:
    game = game_stimuli[game_id]
    played_game_ids.add(game["index"])
    game["stimuli_id"] = game_id
    play_game_objs.append(game)
        

think_human_df = pd.read_csv(constants.THINK_HUMAN_DATA)
# pulling out games according to Ced's game categories
all_game_types = {}
for i, entry in think_human_df.iterrows(): 
    game_type = entry.game_types
    game_id = entry.game_id
    if game_type not in all_game_types: all_game_types[game_type] = {game_id}
    else: all_game_types[game_type].add(game_id)

"""
Load in model preds
"""

model2res = {model: [] for model in ordered_models}
for model, model_pth in model2pth.items(): 
    res = analysis_utils.process_model_runs(model, model_pth, idx2game)
    if res is not None: model2res[model] = res


In [ ]:
'''
Load in pre-run model fits to people

Potentially load from multiple sources to match many different human data paths
'''
model_fit_data = {}

with open("../model-data/play-exp/final_data/heuristics.txt", "r") as f:  # Intuitive Gamer model
    model_fit_data['ours'] = eval(f.readlines()[0])
    

In [ ]:
import pickle

# help from claude

pickle_file_path = "../model-data/play-exp/2025-07-02_heuristic_search_eg.pickle"

with open(pickle_file_path, 'rb') as file:
    loaded_data = pickle.load(file)

model_fit_data['expert'] = loaded_data

In [ ]:
e1 = model_fit_data['expert']


In [ ]:
pickle_file_path = '../model-data/play-exp/final_data/depth3-new-may24.pickle'
with open(pickle_file_path, 'rb') as file:
    loaded_data = pickle.load(file)

model_fit_data['ours-depth3'] = loaded_data

In [ ]:

with open(f"../human-data/play-exp/human-v-human/final-play/final_agg.json", "r") as f: 
    human_gameplay_data = json.load(f)


In [ ]:
all_humans = set()
all_arenas = set()

humans_per_game = {stim: set() for stim in human_gameplay_data} 
arena2players = {}
outcomes_per_game = {}
player2outcomes = {}
player2move_times = {}


show_times = False
overcame_advantage = set()

if show_times: 
    import math 
    n_game_show_rows = 6 
    n_game_show_cols = math.ceil(len(human_gameplay_data) / n_game_show_rows)
    fig, axes=  plt.subplots(n_game_show_rows, n_game_show_cols, figsize=(32,16))
    axes = axes.flatten()
    ax_idx = 0

for stim, entry in human_gameplay_data.items():
    stim_participants = {val for x in entry['order2player'] for val in x.values()}
    humans_per_game[stim] = stim_participants
    all_humans.update(stim_participants)
    all_arenas.update({a for a in entry['arena']})
    
    
    all_move_times = entry['move_times']
    

    max_moves = max(len(game) for game in all_move_times)

    means = []
    std_errors = []
    move_numbers = []
    game_len_avg = np.mean([len(eval(boards)) - 1 for boards in entry['boards']]) 

    for move in range(1, max_moves):
        move_times = [game[move]  for game in all_move_times if len(game) > move and not np.isnan(game[move])]
        if move_times:
            mean_time = np.mean(move_times)
            std_error = analysis_utils.compute_se(move_times) 
            means.append(mean_time)
            std_errors.append(std_error)
            move_numbers.append(move)

    if show_times: 
        ax = axes[ax_idx]
        ax.errorbar(move_numbers, means, yerr=std_errors, fmt='o-', capsize=5, 
                    color='blue', ecolor='gray', markersize=6, linewidth=2)
        ax.axvline(x=game_len_avg, color='r', linestyle='--')

        ax.set_xlabel('Move Number')
        ax.set_ylabel('Time (seconds)')
        n_rows_game, n_cols_game, win_conds = stim.split("*")
        n_rows_game = int(float(n_rows_game))
        n_cols_game = int(float(n_cols_game))
        n_cells = n_rows_game * n_cols_game
        
        game_title_fmt = f"{n_rows_game} x {n_cols_game}\n{analysis_utils.tidy_game_name(win_conds)}"
        ax.set_title(game_title_fmt)
        ax_idx += 1
        
    
    
    
    outcomes = []
    for payoff_data, order2player, outcome, arena, move_times in zip(entry["player_exp_payoffs"], entry["order2player"], entry["outcome"], 
                                                                     entry['arena'], entry['move_times']): 
        
        player2order = {v: k for k,v in order2player.items()}
        game_players = list(player2order)
        if len(game_players) > 0: arena2players.setdefault(arena, set())
       
        if len(payoff_data) != 0: 
            for player in payoff_data: 
                payoff = payoff_data[player]
                order = player2order[player]
                arena2players[arena].add(player)
        else: 
            for player in player2order: 
                arena2players[arena].add(player)
            
        outcome_int = 0  # 0, 1, -1
        if outcome == "Draw":
            outcome_int = 0 
            outcomes.append(outcome_int)
            
        else: 
            outcome_int = int(player2order[outcome]) # winner
            if outcome_int == 2: 
                outcome_int = -1 # second player
            outcomes.append(outcome_int)
            
        for player in game_players: 
            player2outcomes.setdefault(player, [])
            player_order = player2order[player]
            if outcome_int == 0: player_outcome = 0
            else: 
                if outcome == player: player_outcome = 1
                else: player_outcome = -1
            player2outcomes[player].append([player_outcome, stim, player_order])
           
        move_time_vals = move_times[1:] # note: nan is just the empty board at the start (no time) 
        if arena == '01JDQDCK4J093PE7H9JEWZF57X': 
            print(move_times)
            
            count_long = 0
            for i, m in enumerate(move_time_vals): 
                if m > means[i]: count_long += 1
            print("num long: ", count_long/len(move_time_vals))
            print(player2order, stim)
            
        for player, order in player2order.items():
            player2move_times.setdefault(player, [])
            if order == '1':
            # Player X makes the first, third, fifth... moves (even indices in 0-based indexing)
                move_times_player = move_time_vals[::2]
            elif order == '2':
                # Player O makes the second, fourth, sixth... moves (odd indices in 0-based indexing)
                move_times_player = move_time_vals[1::2]
            
            player2move_times[player].append(move_times_player)
                 
            
        
    
    outcomes_per_game[stim] = outcomes
    
if show_times: 
    axes[-1].set_xticks([])
    axes[-1].set_yticks([])
    axes[-1].spines['top'].set_visible(False)
    axes[-1].spines['right'].set_visible(False)
    axes[-1].spines['bottom'].set_visible(False)
    axes[-1].spines['left'].set_visible(False)
    plt.tight_layout()
    plt.savefig(save_dir + "move_times_grid.pdf", dpi=400)
    


In [ ]:
all_agents = ['ours', 
               'ours-depth3',

          'expert', 
          'random']

sub_agents = ['ours', 
               'ours-depth3',

          'expert', 
          'random']

agent2viz = {'ours': 'Novice', 
          'expert': 'Expert', 
          'ours-depth3': 'Novice\n(Depth-3)',
          'expert-depth1': 'Global Eval (Depth-1)',
          'random': 'Random'}

agent2color = {'expert': 'blue', 
          'ours': 'red', 
          'ours-depth3': 'orange',
          'expert-depth1': 'purple',
          'random': 'grey'}

In [ ]:

from scipy.stats import entropy
importlib.reload(models.just_think.run_models)
importlib.reload(analysis_utils)
from models.agents.uni_random import uniform_dist
considered_games = sorted(list(human_gameplay_data))
all_participant_data = {stim: {} for stim in considered_games}

all_model_data = {agent: {stim: {} for stim in considered_games} for agent in all_agents}

# map the game, arena, and move symbol to a particular player
participant_id_map = {}
participant2arena = {}
for agent in all_agents: 

    for idx, (stim, stim_data) in  enumerate(human_gameplay_data.items()): 

        game_id = game2idx[stim]
        game = games[game_id - 1]
        all_game_boards = [eval(board) for board in stim_data["boards"]]
        
        n_cols = len(all_game_boards[0])
        n_rows = len(all_game_boards[0][0])
        n_cells = n_cols * n_rows
        
        for arena_idx, game_boards in enumerate(all_game_boards): 
            arena_name = stim_data["arena"][arena_idx]
            
            order2player = stim_data['order2player'][arena_idx]
            player2order = {player: 'X' if order == '1' else 'O' for order, player in order2player.items()}
            
            for player, order in player2order.items(): 
                if player not in participant_id_map: 
                    participant_id_map[player] = {stim: order, "arena": arena_name}
                else: participant_id_map[player][stim] = order
                
            if len(game_boards) == 0: continue
            starting_board = analysis_utils.convert_board_repr(game_boards[0])
            prev_board = starting_board
            
            arena_data_h = {}
            arena_data_m = {}
                    
            for i, board in enumerate(game_boards[1:]): 

                if agent == "random":
                    move_dist = [uniform_dist(prev_board)]

                else: 
                    try:
                        output = model_fit_data[agent][arena_name][str(game['index'])][str(i)]

                        move_dist = output['dist']

                        if move_dist is None: 
                            print("NONE: ", agent, i, stim, arena)
                        
                        if type(move_dist) != list:

                            move_dist = [move_dist] 
                        m_outputs = []
                        for m_output in move_dist: 
                            new_dict = {}
                            for k, v in m_output.items():
                                if v != -np.inf: new_dict[k] = v
                            m_outputs.append(new_dict)
                        
                        move_dist = m_outputs
                        
                        
                    except Exception as e:
                        print("error: ", e, agent, stim, arena_name, str(i+1), str(game['index']), model_fit_data[agent][arena_name].keys()) 
                        continue 
                                
                round_board = analysis_utils.convert_board_repr(board)
                # what was this player's next move
                diff = np.array(round_board) != np.array(prev_board)  # find where the boards differ
                row, col = np.where(diff)  # get the indices of the differences
                # just one move
                move = (int(row[0]), int(col[0]))
                player_id = round_board[move[0]][move[1]]
                num_legal = np.sum(np.array(prev_board) == '_')
                
                arena_data_h[i] = (move, player_id)
                arena_data_m[i] = (move_dist, num_legal)
                
                
                
                prev_board = round_board
                
                
            all_participant_data[stim][arena_name] = arena_data_h
            all_model_data[agent][stim][arena_name] = arena_data_m

In [ ]:
len(considered_games)

In [ ]:

importlib.reload(analysis_utils)
metrics = [

            'topK-1',
            'topK-3', 
            'topK-5', 

           'rank',
           'tiecounts', 

           ]
metric_data = {
    'game': [],
    'arena': [],
    'source': []
}

metric_data.update({metric: [] for metric in metrics})
n_repeat_sample = 100
np.random.seed(7)
random.seed(7)

dist_ents = {}


for agent, m_data in all_model_data.items(): 

    for stim, stim_data in all_participant_data.items(): 
        stim_scores = []
        for arena, arena_data in stim_data.items(): 
            for i, (move, player_id) in arena_data.items(): 
                m_dists, n_legal = m_data[stim][arena][i]
                if m_dists is None: 
                    continue
                for metric_idx, metric in enumerate(metrics): 
                    
                    if metric == 'pmove': 
                        norm_mdists = [analysis_utils.softmax_dist(m_dist, T=1) for m_dist in m_dists]
                        scores = np.mean([analysis_utils.get_pmove(norm_mdist, move) for norm_mdist in norm_mdists]) #for dist in dists]
                    
                    elif metric == 'acc': 
                        scores = np.mean([analysis_utils.get_acc(m_dist, move) for m_dist in m_dists])
                    
                    elif 'topK' in metric:
                        k_thresh = int(metric.split("topK-")[-1])

                        all_scores = []
                        for _ in range(n_repeat_sample):
                        
                            scores = np.mean([analysis_utils.get_acc_topK(m_dist, move,K=k_thresh, ties="sample") for m_dist in m_dists])
                            all_scores.append(scores)
                        scores = np.mean(all_scores)
                    elif metric == 'rank' or metric == 'tiecounts': 
                        ranks = []
                        same_rank_counts = []
                        for m_dist in m_dists: 
                            rank, same_rank_count = analysis_utils.get_rank(m_dist, move)
                            ranks.append(rank)
                            same_rank_counts.append(same_rank_count)
                        if metric == 'rank': scores = np.mean(ranks)
                        if metric == 'tiecounts': scores = np.mean(same_rank_counts)
                        
                    elif 'rel-prob' in metric: 
                        rel_probs = []
                        adj_rel_probs=  []
                        for m_dist in m_dists: 
                            norm_mdist = analysis_utils.softmax_dist(m_dist, T=1)
                            pmove = analysis_utils.get_pmove(norm_mdist, move)
                            maxp = np.max(list(norm_mdist.values()))
                            rel_prob = pmove - maxp
                            rel_probs.append(rel_prob)
                            if np.sum([pmove == val for val in norm_mdist.values()]) == 0: 
                                print("ERROR", agent)
                                print(pmove, move in norm_mdist, norm_mdist.keys(), move)
                                print(move, stim)
                                same_score = 0
                                dist_ent = 0
                            else: 
                                dist_ent = entropy(list(norm_mdist.values()))
                                same_score = np.sum([pmove == val for val in norm_mdist.values()]) - 1
                            dist_ents.setdefault(agent, [])
                            dist_ents[agent].append(dist_ent)
                            adj_rel_probs.append(dist_ent)
                            
                        if metric == 'rel-prob': scores = np.mean(rel_probs)
                        if metric == 'rel-prob-occ': scores = np.mean(adj_rel_probs)
                    
                    if metric_idx == 0: 

                        metric_data['game'].append(stim)
                        metric_data['arena'].append(arena)
                        metric_data['source'].append(agent2viz[agent])

                    metric_data[metric].append(scores)
                        
metric_df = pd.DataFrame.from_dict(metric_data)
metric_df.to_csv(f"{main_final_save_dir}/acc_df.csv")


metric2viz = {'coverage': 'Coverage',
              'rank': 'Rank', 'acc': 'Top-1 Acc', 'pmove': 'P(move) played',
              'topK-1': 'Top-1 Acc',
              'topK-3': 'Top-3 Acc',
              'topK-5': 'Top-5 Acc',
              'topK-10': 'Top-10 Acc',
              'tiecounts': 'Same Rank',
              'rel-prob': 'Relative P(Move)',
              'rel-prob-occ': 'Adj Relative P(Move)'
              }


In [ ]:
''' 
Vary slop params
'''
import admixture_utils as pf_utils
import gc
import random
importlib.reload(pf_utils)
np.random.seed(7)
random.seed(7)
step = 0.05
max_alpha_val = 1.0-(1e-10)
alpha_vals= list(np.arange(0.5, max_alpha_val, step))

arena_subset_prop = -1
game_subset_prop = -1
n_bootstrap_samples = 1
fixed_alpha_res = {}
game_stage = 'all'
viz_agents = all_agents
for agent in viz_agents:
    fixed_alpha_res.setdefault(agent, {alpha: [] for alpha in alpha_vals})
    for alpha in alpha_vals: 
        for j in range(n_bootstrap_samples): 
            print("on trial: ", j, alpha, agent)
            score, all_scores, _, game_ordering = pf_utils.score_fixed_alpha(agent,
                                                                                         all_model_data, 
                                                                                         all_participant_data, 
                                                                                         alpha=alpha,
                            arena_subset_prop=arena_subset_prop, game_subset_prop=game_subset_prop,
                            game_stage=game_stage, temperature = 1)
            fixed_alpha_res[agent][alpha].append([all_scores,game_ordering])
            print("done run")
            gc.collect()


In [ ]:
with open(f"{main_final_save_dir}agg_scores_alpha_NEW.json", "w") as f: 
    json.dump(fixed_alpha_res, f)


In [ ]:

import admixture_utils as pf_utils
importlib.reload(pf_utils)
import random
np.random.seed(7)
random.seed(7)

step = 0.05
max_alpha_val = 1.0-(1e-10)
alpha_vals= list(np.arange(0.5, max_alpha_val, step))
arena_subset_prop = -1
game_subset_prop = -1
n_bootstrap_samples = 1

game_stages = ['early', 'middle', 'late']
viz_agents = sub_agents

stage_res = {game_stage: {} for game_stage in game_stages}
for game_stage in game_stages: 
    fixed_alpha_res = {}
    
    for agent in viz_agents: 
        
        fixed_alpha_res.setdefault(agent, {alpha: [] for alpha in alpha_vals})
        for alpha in alpha_vals: 
            for _ in range(n_bootstrap_samples): 
                score, all_scores, _, game_ordering = pf_utils.score_fixed_alpha(agent,
                                                                                            all_model_data, 
                                                                                            all_participant_data, 
                                                                                            alpha=alpha,
                                arena_subset_prop=arena_subset_prop, game_subset_prop=game_subset_prop,
                                game_stage=game_stage)
                fixed_alpha_res[agent][alpha].append([all_scores, game_ordering])
                
    stage_res[game_stage] = fixed_alpha_res
    
with open(f"{main_final_save_dir}game_stages_alpha_NEW.json", "w") as f: 
    json.dump(stage_res, f)

In [ ]:
'''
Admixture per game
'''
import admixture_utils as pf_utils
importlib.reload(pf_utils)
import random
np.random.seed(7)
random.seed(7)

arena_subset_prop = 0.8
game_subset_prop = -1
n_bootstrap_samples = 5 

temp_step = 0.5
temp_vals= np.arange(0.5, 3.0 + temp_step, temp_step)
viz_agents = ['ours', 'expert', 'random']
admixture_agents = [agent for agent in viz_agents if agent != 'random'] # random automatically included

game_bootstrap_res_temp = {}

for i, game in enumerate(all_participant_data): 
    print("game: ", game)

    game_bootstrap_res_temp.setdefault(game, {temp: [] for temp in temp_vals})

    for temp in temp_vals: 
        bootres = []
        for _ in range(n_bootstrap_samples):
            res = pf_utils.optimize_mixture_weights(all_model_data, all_participant_data,
                                                    model_names=admixture_agents,
                    arena_subset_prop=arena_subset_prop, game_id=game,
                     temperature=temp)
            bootres.append(res)
        game_bootstrap_res_temp[game][temp] = bootres
with open(f"{main_final_save_dir}per_game_admixture.json", "w") as f: 
    json.dump(game_bootstrap_res_temp, f)

In [ ]:
'''
Admixture
'''
import admixture_utils as pf_utils
importlib.reload(pf_utils)
import random
np.random.seed(7)
random.seed(7)


arena_subset_prop = -1 
game_subset_prop = -1
n_bootstrap_samples = 1 

temp_step = 0.5
temp_vals= np.arange(0.5, 3.0 + temp_step, temp_step)
viz_agents = ['ours', 'expert', 'random']
admixture_agents = [agent for agent in viz_agents if agent != 'random'] # random automatically included

participant_bootstrap_res_temp = {}

for participant_id, player_data in participant_id_map.items():
    participant_bootstrap_res_temp.setdefault(participant_id, {temp: [] for temp in temp_vals})
    
    
    arena_id = player_data['arena']

    for temp in temp_vals: 
        participant_res = []
        for _ in range(n_bootstrap_samples):
            res = pf_utils.optimize_mixture_weights(all_model_data, all_participant_data,
                                                    model_names=admixture_agents,
                    arena_id=arena_id, game_subset_prop=game_subset_prop,
                        participant_play_order=player_data, temperature=temp)
            participant_res.append(res)
        participant_bootstrap_res_temp[participant_id][temp] = participant_res
        
with open(f"{main_final_save_dir}per_participant_admixture_individ.json", "w") as f: 
    json.dump(participant_bootstrap_res_temp, f)